# Prompting

Prompting means giving a language model instructions, questions, or context to guide its response.

## Messages and roles

Chat models receive a conversation as a sequence of messages. Each message has a role and content.

- **System:** Sets instructions for how the model should behave, such as its tone or response format.
- **User:** Contains the user's question, instructions, or other input.
- **Assistant:** Contains a model response or an example response supplied as conversation history.

These are message roles, rather than prompting techniques such as zero-shot or few-shot prompting.

In LangChain, `human` corresponds to `user`, and `ai` corresponds to `assistant`.


## Message roles in practice

This example uses Qwen2.5-Instruct with a system instruction to answer briefly, like a pirate. We provide a user–assistant exchange as conversation history, then ask a follow-up question to demonstrate how the model uses that context.


In [1]:
import gc
import torch
from IPython.display import Markdown, display
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
if "model" in globals():
    globals().pop("model")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

messages = [
    {
        "role": "system",
        "content": (
            "Answer briefly, like a pirate. "
            "If you lack up-to-date information, say so rather than guess."
        ),
    },
    {"role": "user", "content": "What state is Atlanta in?"},
    {"role": "assistant", "content": "Arr, Atlanta be in Georgia."},
    {"role": "user", "content": "What is the weather in Atlanta right now?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
    )

response_ids = output_ids[0, inputs["input_ids"].shape[1]:]
response = tokenizer.decode(response_ids, skip_special_tokens=True)

display(Markdown("### Model’s response"))
print(response)


/home/nick/github-projects/prompting-base-models/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1091.44it/s]


### Model’s response

I'm sorry, but I don't have real-time data access to provide current conditions for Atlanta. You might want to check a reliable weather website or app for the most accurate and up-to-date information.


### About the output

The system message asks the model to answer briefly, like a pirate, and acknowledge when it lacks up-to-date information. The supplied user–assistant exchange provides conversation history, and the final user message asks about the current weather in Atlanta.

The model has no current weather data in this example. Its response shows whether it acknowledges that limitation while following the requested style.


## Reasoning Models

Reasoning models can generate intermediate reasoning steps before producing a final answer, using additional tokens and computation.

The next example uses Qwen3-0.6B, which supports thinking and non-thinking modes. In thinking mode, it generates thinking content within `<think>...</think>` tags, followed by its final answer.

![Example of a reasoning model response](../assets/images/reasoning-example.png)

Chain-of-thought prompting encourages intermediate reasoning through instructions or worked examples. It is a prompting technique; the tags above are part of Qwen3’s output format.

References: [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), [Chain-of-Thought Prompting](https://arxiv.org/abs/2201.11903).


## Qwen3 in thinking mode

We enable thinking mode and ask Qwen3-0.6B to calculate `17 × 24`. The code separates its generated thinking content from the final answer so we can inspect both.

Afterward, we can disable thinking mode and compare the responses.


In [2]:
model_name = "Qwen/Qwen3-0.6B"

if "model" in globals():
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

messages = [{"role": "user", "content": "What is 17 * 24?"}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    enable_thinking=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=2048,
        do_sample=True,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
    )

response_ids = output_ids[0, inputs["input_ids"].shape[1]:]
response = tokenizer.decode(response_ids, skip_special_tokens=True)

thinking, separator, answer = response.partition("</think>")

if separator:
    thinking = thinking.removeprefix("<think>").strip()

    display(Markdown("### Model-generated thinking"))
    print(thinking)

    display(Markdown("### Model’s final answer"))
    print(answer.strip())
else:
    if output_ids[0, -1].item() != tokenizer.eos_token_id:
        display(Markdown("### Thinking cut off — no final answer generated"))
    else:
        display(Markdown("### Model’s response"))
    print(response.removeprefix("<think>").strip())


Loading weights: 100%|██████████| 311/311 [00:01<00:00, 240.18it/s]


### Thinking cut off — no final answer generated

Okay, so I need to find 17 multiplied by 24. Hmm, let me think. I remember that multiplying two numbers can be done by breaking them down. Let me see... 17 and 24. Maybe I can use the standard multiplication algorithm here. Let me write it out.

First, 17 multiplied by 24. To multiply these, I can think of 17 as 10 + 7, and 24 as 20 + 4. Then, I can use the distributive property to multiply each part separately and then add the results together. Let me try that.

So, 17 times 24 is equal to (10 + 7) multiplied by (20 + 4). Using the distributive property, that would be 10*20 + 10*4 + 7*20 + 7*4. Let me calculate each term:

10*20 is 200. 10*4 is 40. 7*20 is 140. 7*4 is 28. Now adding all those together: 200 + 40 is 240, plus 140 is 380, plus 28 is 408. So, 17*24 equals 408. Wait, but let me check if there's another way to do this to confirm.

Alternatively, I can use the standard multiplication method. Let me try that. Multiplying 17 by 24. Let's align them:

   17
x24
-----
First, mul

### About the output

The **Model-generated thinking** section shows the intermediate reasoning generated by the model. The **Model’s final answer** section shows its final response. Headings are rendered as Markdown, while generated content is displayed as plain text.

Both sections count toward the 2,048-token generation limit. If generation reaches that limit before `</think>`, the output is labeled **Thinking cut off — no final answer generated**. The final answer can also be cut short if the limit is reached after the closing tag. A truncated response is not necessarily an arithmetic error.

To compare modes, change `enable_thinking=True` to `False` and rerun the cell. A completed response without a closing `</think>` tag appears under **Model’s response**.
